<a href="https://colab.research.google.com/github/clara-eng/Assignment-1-/blob/main/Exam_time_table_agent_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import random
from collections import defaultdict

# Constants
HARD_SUBJECTS = ['Math', 'Physics']
MAX_EXAMS_PER_DAY = 5
MIN_EXAM_DAYS = 5
# global variable
no_improvement_counter = 0


# # Load data with error handling
# def load_csv(path, name):
#     try:
#         return pd.read_csv(path)
#     except FileNotFoundError:
#         print(f"[ERROR] File not found: {name}")
#         return pd.DataFrame()
def load_csv(path, name):
    try:
        df = pd.read_csv(path)
        if df.empty:
            print(f"[WARNING] Empty dataframe loaded from {name}")
        return df
    except FileNotFoundError:
        print(f"[ERROR] File not found: {name}")
        return pd.DataFrame()
    except Exception as e:
        print(f"[ERROR] Could not load {name}: {str(e)}")
        return pd.DataFrame()

# Load all required datasets
import pandas as pd

schedule_df = pd.read_csv("schedule.csv", header=0)
print("Schedule columns:", schedule_df.columns.tolist())  # دي عشان تتأكد

# اختبار إضافي للتأكد
if 'timeslot_id' not in schedule_df.columns:
    print("⚠️ 'timeslot_id' column is missing or misnamed!")
else:
    print("✅ 'timeslot_id' column exists.")

timeslots_df = pd.read_csv("timeslots.csv", header=0)  # نفس الفكرة لو بتحب تتأكد برضو

# بعدها كمل الكود زي ما هو:
schedule_df = schedule_df.merge(timeslots_df, on="timeslot_id", how="left")

students_df = load_csv("students.csv", "students")
courses_df = load_csv("courses.csv", "courses")
classrooms_df = load_csv("classrooms.csv", "classrooms")
timeslots_df = load_csv("timeslots.csv", "timeslots")

# Merge timeslot and classroom info into schedule_df
schedule_df = schedule_df.merge(timeslots_df, on="timeslot_id", how="left")
schedule_df = schedule_df.merge(classrooms_df, on="classroom_id", how="left")

# Build student-to-courses and course-to-students maps
student_courses = schedule_df.groupby('student_id')['course_id'].apply(list).to_dict()
course_students = schedule_df.groupby('course_id')['student_id'].apply(list).to_dict()
timeslot_to_day = dict(zip(timeslots_df['timeslot_id'], timeslots_df['day']))

# === Metadata Extraction ===
exams = courses_df['course_id'].tolist()
rooms = classrooms_df['classroom_id'].tolist()
timeslots = timeslots_df['timeslot_id'].tolist()
courses = courses_df['course_id'].unique()
course_names = dict(zip(courses_df['course_id'], courses_df['course_name']))


# === Helper Functions ===
def get_course_students(course_id):
    if isinstance(course_id, (list, tuple, np.ndarray)):
        raise ValueError(f"Expected single course_id, got {course_id}")
    return schedule_df[schedule_df['course_id'] == course_id]['student_id'].tolist()


def get_room_capacity(classroom_id):
    return classrooms_df[classrooms_df['classroom_id'] == classroom_id]['capacity'].values[0]


def is_hard_subject(course_id):
    name = courses_df.loc[courses_df['course_id'] == course_id, 'course_name'].values[0]
    return any(subj in name for subj in HARD_SUBJECTS)


def find_available_room(time, timetable):
    """
    تبحث عن قاعة متاحة في وقت محدد
    :param time: الوقت المطلوب
    :param timetable: الجدول الزمني الحالي
    :return: اسم القاعة المتاحة أو None إذا لم توجد
    """
    # 1. الحصول على القاعات المستخدمة في هذا الوقت
    used_rooms = {room for (t, room) in timetable.values() if t == time}

    # 2. البحث عن القاعات غير المستخدمة مرتبة حسب السعة (الأكبر أولاً)
    available_rooms = classrooms_df[~classrooms_df['classroom_id'].isin(used_rooms)]
    available_rooms = available_rooms.sort_values('capacity', ascending=False)

    # 3. إرجاع أول قاعة متاحة (إن وجدت)
    return available_rooms.iloc[0]['classroom_id'] if not available_rooms.empty else None


# === NEW: Find multiple rooms to fit students ===
def find_multiple_rooms(time, required_capacity, timetable):
    available_rooms = []
    used_rooms = {room for (t, room) in timetable.values() if t == time}

    for _, row in classrooms_df.iterrows():
        room_id = row['classroom_id']
        if room_id not in used_rooms:
            available_rooms.append((room_id, row['capacity']))

    # Try to accumulate enough capacity
    selected_rooms = []
    total = 0
    for room_id, cap in sorted(available_rooms, key=lambda x: -x[1]):  # big rooms first
        selected_rooms.append(room_id)
        total += cap
        if total >= required_capacity:
            return selected_rooms
    return None  # Not enough capacity


def print_timetable(timetable):
    # Create a matrix (list of lists) for the timetable representation
    timetable_matrix = [["" for _ in range(24)] for _ in range(len(classrooms_df))]

    # Fill in the timetable matrix
    for course, (time, room) in timetable.items():
        room_index = classrooms_df[classrooms_df['classroom_id'] == room].index[0]
        course_name = course_names.get(course, "Unknown Course")
        timetable_matrix[room_index][time - 1] = f"Course {course} ({course_name})"

    # Print the timetable in a readable format
    print(f"{'Room':<10} {'Time':<10} {'Day':<10} {'Course Name'}")
    for room_index, room_row in enumerate(timetable_matrix):
        room_id = classrooms_df.iloc[room_index]['classroom_id']
        for time_index, entry in enumerate(room_row):
            if entry:
                timeslot_id = time_index + 1
                day = timeslot_to_day.get(timeslot_id, "Unknown")
                print(f"{room_id:<10} {timeslot_id:<10} {day:<10} {entry}")


# === Repair Room Conflicts ===
def repair_room_conflicts(timetable):
    room_usage = defaultdict(set)
    for course, (time, room) in timetable.items():
        if (time, room) in room_usage:
            new_room = find_available_room(time, timetable)
            if new_room:
                timetable[course] = (time, new_room)
            else:
                print(f"No available rooms for course {course} at time {time}.")
        room_usage[(time, room)].add(course)
    return timetable


# === Heuristic-Based Initialization ===
def create_heuristic_individual():
    timetable = {}
    sorted_courses = sorted(courses, key=lambda c: len(get_course_students(c)), reverse=True)

    for course in sorted_courses:
        if course in timetable:  # Skip if course is already assigned
            continue

        assigned = False
        # Track usage of timeslots to sort by least used
        timeslot_load = defaultdict(int)
        for _, (time, _) in timetable.items():
            timeslot_load[time] += 1
        sorted_timeslots = sorted(timeslots, key=lambda t: timeslot_load[t])

        for time in sorted_timeslots:
            for room in classrooms_df['classroom_id']:
                room_capacity = get_room_capacity(room)
                students_in_course = get_course_students(course)
                conflict = any(time == timetable.get(other_course, (None, None))[0] and
                               bool(set(get_course_students(other_course)) & set(students_in_course))
                               for other_course in timetable)
                if len(students_in_course) <= room_capacity and (time, room) not in timetable.values() and not conflict:
                    timetable[course] = (time, room)
                    assigned = True
                    break
                else:
                    # Try multiple rooms if needed
                    multi_rooms = find_multiple_rooms(time, len(students_in_course), timetable)
                    if multi_rooms and not conflict:
                        timetable[course] = (time, multi_rooms[0])  # ✅ Assign first available room
                        assigned = True
                        break
            if assigned:
                break

        if not assigned:
            print(f"[WARNING] Could not assign course {course} with strict constraints. Trying fallback...")
            students_in_course = get_course_students(course)
            for time in sorted_timeslots:
                for room in classrooms_df['classroom_id']:
                    room_capacity = get_room_capacity(room)
                    if len(students_in_course) <= room_capacity:
                        timetable[course] = (time, room)
                        print(
                            f"[Fallback] Assigned course {course} to time {time} and room {room} (with possible conflict).")
                        assigned = True
                        break
                if assigned:
                    break
            if not assigned:
                print(f"[ERROR] Failed to assign course {course} even with fallback.")

    return repair_room_conflicts(timetable)


# === Constraint-Satisfying Initialization ===
def create_constraint_satisfying_individual():
    timetable = {}
    for course in courses:
        assigned = False

        # Sort timeslots by how loaded they are
        timeslot_load = defaultdict(int)
        for _, (time, _) in timetable.items():
            timeslot_load[time] += 1
        sorted_timeslots = sorted(timeslots, key=lambda t: timeslot_load[t])

        for time in sorted_timeslots:
            for room in classrooms_df['classroom_id']:
                students_in_course = get_course_students(course)
                room_capacity = get_room_capacity(room)
                conflict = any(time == timetable.get(other_course, (None, None))[0] and
                               bool(set(get_course_students(other_course)) & set(students_in_course))
                               for other_course in timetable)
                if len(students_in_course) <= room_capacity and (time, room) not in timetable.values() and not conflict:
                    timetable[course] = (time, room)
                    assigned = True
                    break
                else:
                    # Try multiple rooms if needed
                    multi_rooms = find_multiple_rooms(time, len(students_in_course), timetable)
                    if multi_rooms and not conflict:
                        timetable[course] = (time, multi_rooms[0])  # ✅ Assign first available room
                        assigned = True
                        break
            if assigned:
                break

        if not assigned:
            print(f"[WARNING] Could not assign course {course} with strict constraints. Trying fallback...")
            students_in_course = get_course_students(course)
            for time in sorted_timeslots:
                for room in classrooms_df['classroom_id']:
                    room_capacity = get_room_capacity(room)
                    if len(students_in_course) <= room_capacity:
                        timetable[course] = (time, room)
                        print(
                            f"[Fallback] Assigned course {course} to time {time} and room {room} (with possible conflict).")
                        assigned = True
                        break
                if assigned:
                    break
            if not assigned:
                print(f"[ERROR] Failed to assign course {course} even with fallback.")

    return repair_room_conflicts(timetable)


# === Integer Representation ===
def create_integer_individual():
    return {course: random.choice(timeslots) for course in courses}


# === Build Timetable from Integer ===
# === Build Timetable from Integer ===
def build_timetable_from_integer(integer_representation):
    timetable = {}
    num_courses = len(integer_representation)  # عدد المواد الدراسية

    for i, course in enumerate(integer_representation):
        assigned = False
        for time in timeslots:
            for room in classrooms_df['classroom_id']:
                students_in_course = get_course_students(course)
                conflict = any(time == timetable.get(other_course, (None, None))[0] and
                               bool(set(get_course_students(other_course)) & set(students_in_course))
                               for other_course in timetable)
                if (time, room) not in timetable.values() and not conflict:
                    timetable[course] = (time, room)
                    assigned = True
                    break
            if assigned:
                break
        if not assigned:
            print(f"Could not assign course {course}")

    return repair_room_conflicts(timetable)

# === Permutation Representation ===
def create_permutation_individual():
    course_list = list(courses)
    random.shuffle(course_list)
    return course_list


# === Build Timetable from Permutation ===
def build_timetable_from_permutation(permutation):
    timetable = {}
    for course in permutation:
        assigned = False
        students_in_course = get_course_students(course)  # نحركها خارج الحلقات لتجنب تكرار الاستدعاء

        # نرتب الأوقات بناء على الأقل استخداماً أولاً
        timeslot_load = defaultdict(int)
        for (time, _) in timetable.values():
            timeslot_load[time] += 1
        sorted_timeslots = sorted(timeslots, key=lambda t: timeslot_load[t])

        for time in sorted_timeslots:  # نستخدم الأوقات المرتبة
            # نرتب القاعات بالأكبر سعة أولاً
            sorted_rooms = sorted(classrooms_df.to_dict('records'),
                                  key=lambda x: -x['capacity'])

            for room_data in sorted_rooms:
                room = room_data['classroom_id']
                capacity = room_data['capacity']

                # 1. التحقق من سعة القاعة
                if len(students_in_course) > capacity:
                    continue

                # 2. التحقق من عدم حجز القاعة في نفس الوقت
                if (time, room) in [(t, r) for (t, r) in timetable.values()]:
                    continue

                # 3. التحقق من عدم تعارض الطلاب
                conflict = False
                for other_course, (other_time, other_room) in timetable.items():
                    if time == other_time:
                        other_students = get_course_students(other_course)
                        if set(students_in_course) & set(other_students):
                            conflict = True
                            break

                if not conflict:
                    timetable[course] = (time, room)
                    assigned = True
                    break

            if assigned:
                break

        if not assigned:
            # محاولة ثانية مع تخفيف القيود (النسخة الاحتياطية)
            for time in timeslots:
                for room in classrooms_df['classroom_id']:
                    if (time, room) not in [(t, r) for (t, r) in timetable.values()]:
                        timetable[course] = (time, room)
                        print(f"[Warning] Assigned course {course} with possible conflicts")
                        assigned = True
                        break
                if assigned:
                    break

            if not assigned:
                print(f"[Error] Failed to assign course {course} after all attempts")

    return repair_room_conflicts(timetable)

# def hybrid_fitness(timetable):
#     hard_penalty = 0
#     soft_penalty = 0
#
#     student_schedule = defaultdict(list)
#     student_exam_courses = defaultdict(list)
#     room_usage = defaultdict(int)
#     time_room_set = set()
#     exam_days = defaultdict(list)
#
#     total_room_capacity = sum(get_room_capacity(room['classroom_id']) for room in classrooms_df.to_dict('records'))
#
#     # === Constraint Checks ===
#     for course, (time, room) in timetable.items():
#         # print("Debug - course:", course)
#         students = get_course_students(course)
#         capacity = get_room_capacity(room)
#
#         # Room Capacity Violation
#         if len(students) > capacity:
#             hard_penalty += (len(students) - capacity) * 5
#
#         # Room Double Booking
#         if (time, room) in time_room_set:
#             hard_penalty += 10
#         else:
#             time_room_set.add((time, room))
#
#         room_usage[(time, room)] += len(students)
#
#         for s in students:
#             student_schedule[s].append(time)
#             student_exam_courses[s].append((time, course))
#
#         exam_days[time // 5].append(course)
#
#     # Room Overuse
#     for (time, room), usage in room_usage.items():
#         if usage > get_room_capacity(room):
#             hard_penalty += 10
#
#     # Room Utilization
#     total_used_capacity = sum(min(usage, get_room_capacity(room)) for (time, room), usage in room_usage.items())
#     utilization = total_used_capacity / total_room_capacity if total_room_capacity > 0 else 0
#     if utilization < 0.5:
#         soft_penalty += 20
#
#     # Student Conflicts and Gaps
#     for student, times in student_schedule.items():
#         if len(times) != len(set(times)):
#             hard_penalty += 10 * (len(times) - len(set(times)))
#
#         sorted_times = sorted(times)
#         for i in range(1, len(sorted_times)):
#             gap = sorted_times[i] - sorted_times[i - 1]
#             if gap == 0:
#                 soft_penalty += 5
#             elif gap == 1:
#                 soft_penalty += 2
#
#     # Difficult Exam Spacing, Distribution, Daily Limits
#     for student, exam_list in student_exam_courses.items():
#         sorted_exams = sorted(exam_list)
#         gaps = [sorted_exams[i][0] - sorted_exams[i - 1][0] for i in range(1, len(sorted_exams))]
#
#         # Even Distribution
#         if gaps:
#             avg_gap = sum(gaps) / len(gaps)
#             if avg_gap < 1:
#                 soft_penalty += 5
#
#         # Back-to-back difficult exams
#         for i in range(1, len(sorted_exams)):
#             t1, c1 = sorted_exams[i - 1]
#             t2, c2 = sorted_exams[i]
#             gap = t2 - t1
#             if is_hard_subject(c1) and is_hard_subject(c2):
#                 if gap < 2:
#                     soft_penalty += 6
#                 if t1 // 5 == t2 // 5:
#                     soft_penalty += 5  # same day
#
#         # Multi-subject fatigue
#         if len(sorted_exams) >= 3:
#             for i in range(1, len(sorted_exams)):
#                 if sorted_exams[i][0] - sorted_exams[i - 1][0] <= 1:
#                     soft_penalty += 3
#
#     # Exams Per Day Limit
#     for day, exams in exam_days.items():
#         if len(exams) > MAX_EXAMS_PER_DAY:
#             soft_penalty += (len(exams) - MAX_EXAMS_PER_DAY) * 3
#
#     # Minimum Spread
#     if len(exam_days) < MIN_EXAM_DAYS:
#         soft_penalty += (MIN_EXAM_DAYS - len(exam_days)) * 2
#
#     # Exam Time Variance
#     all_exam_times = [time for times in student_schedule.values() for time in times]
#     if all_exam_times:
#         mean_time = sum(all_exam_times) / len(all_exam_times)
#         variance = sum((time - mean_time) ** 2 for time in all_exam_times) / len(all_exam_times)
#         if variance < 1:
#             soft_penalty += 15
#
#     # Final fitness score
#     return - (1000 * hard_penalty + soft_penalty)
def hybrid_fitness(timetable):
    # Initialize penalties and tracking structures
    penalties = {
        'hard': 0,
        'soft': 0
    }

    # Tracking structures
    trackers = {
        'student_schedule': defaultdict(list),
        'student_exams': defaultdict(list),
        'room_usage': defaultdict(int),
        'time_room_set': set(),
        'exam_days': defaultdict(list)
    }

    # Constants
    CONSTRAINTS = {
        'room_capacity_violation': 5,
        'double_booking': 10,
        'room_overuse': 10,
        'low_utilization': 20,
        'student_conflict': 10,
        'same_day_hard_exams': 5,
        'back_to_back_hard': 6,
        'multi_exam_fatigue': 3,
        'exams_per_day_limit': 3,
        'min_spread': 2,
        'low_variance': 15
    }

    # Helper functions
    def calculate_variance(times):
        mean = sum(times) / len(times)
        return sum((t - mean) ** 2 for t in times) / len(times)

    # Main constraint checks
    total_capacity = sum(get_room_capacity(room['classroom_id'])
                         for room in classrooms_df.to_dict('records'))

    for course, (time, room) in timetable.items():
        students = get_course_students(course)
        capacity = get_room_capacity(room)

        # Room constraints
        if len(students) > capacity:
            penalties['hard'] += (len(students) - capacity) * CONSTRAINTS['room_capacity_violation']

        if (time, room) in trackers['time_room_set']:
            penalties['hard'] += CONSTRAINTS['double_booking']
        else:
            trackers['time_room_set'].add((time, room))

        trackers['room_usage'][(time, room)] += len(students)

        # Student tracking
        for student in students:
            trackers['student_schedule'][student].append(time)
            trackers['student_exams'][student].append((time, course))

        trackers['exam_days'][time // 5].append(course)

    # Room overuse check
    for (time, room), usage in trackers['room_usage'].items():
        if usage > get_room_capacity(room):
            penalties['hard'] += CONSTRAINTS['room_overuse']

    # Utilization check
    used_capacity = sum(min(usage, get_room_capacity(room))
                        for (time, room), usage in trackers['room_usage'].items())
    utilization = used_capacity / total_capacity if total_capacity > 0 else 0
    if utilization < 0.5:
        penalties['soft'] += CONSTRAINTS['low_utilization']

    # Student constraints
    for student, times in trackers['student_schedule'].items():
        unique_times = set(times)
        if len(times) != len(unique_times):
            penalties['hard'] += CONSTRAINTS['student_conflict'] * (len(times) - len(unique_times))

        sorted_times = sorted(times)
        for i in range(1, len(sorted_times)):
            gap = sorted_times[i] - sorted_times[i - 1]
            penalties['soft'] += CONSTRAINTS['back_to_back_hard'] if gap < 2 else 0

    # Exam difficulty constraints
    for student, exams in trackers['student_exams'].items():
        sorted_exams = sorted(exams)

        # Hard exams spacing
        for i in range(1, len(sorted_exams)):
            t1, c1 = sorted_exams[i - 1]
            t2, c2 = sorted_exams[i]
            if is_hard_subject(c1) and is_hard_subject(c2):
                if t2 - t1 < 2:
                    penalties['soft'] += CONSTRAINTS['back_to_back_hard']
                if t1 // 5 == t2 // 5:
                    penalties['soft'] += CONSTRAINTS['same_day_hard_exams']

        # Multi-exam fatigue
        if len(sorted_exams) >= 3:
            for i in range(1, len(sorted_exams)):
                if sorted_exams[i][0] - sorted_exams[i - 1][0] <= 1:
                    penalties['soft'] += CONSTRAINTS['multi_exam_fatigue']

    # Global constraints
    for day, exams in trackers['exam_days'].items():
        if len(exams) > MAX_EXAMS_PER_DAY:
            penalties['soft'] += (len(exams) - MAX_EXAMS_PER_DAY) * CONSTRAINTS['exams_per_day_limit']

    if len(trackers['exam_days']) < MIN_EXAM_DAYS:
        penalties['soft'] += (MIN_EXAM_DAYS - len(trackers['exam_days'])) * CONSTRAINTS['min_spread']

    # Time variance calculation

    all_times = [t for times in trackers['student_schedule'].values() for t in times]
    if all_times and calculate_variance(all_times) < 1:
        penalties['soft'] += CONSTRAINTS['low_variance']
    # print(f"العقوبات الصارمة: {penalties['hard']}، العقوبات المرنة: {penalties['soft']}")
    return -((10 * penalties['hard']) + penalties['soft'])

# === Population Initialization ===
# population_size = 100
# population = []
#
# for _ in range(population_size // 2):
#     timetable = create_heuristic_individual()
#     population.append(timetable)
#
# for _ in range(population_size // 2):
#     timetable = create_constraint_satisfying_individual()
#     population.append(timetable)
#
# # Generate initial populations
# heuristic_population = [create_heuristic_individual() for _ in range(10)]
# constraint_satisfying_population = [create_constraint_satisfying_individual() for _ in range(10)]
# population = heuristic_population + constraint_satisfying_population


# في بداية الكود مع الثوابت الأخرى
INITIAL_POPULATION_SIZE = 100
INITIAL_HEURISTIC_RATIO = 0.5  # 50% heuristic, 50% constraint-based

# لاحقاً عند التهيئة
heuristic_count = int(INITIAL_POPULATION_SIZE * INITIAL_HEURISTIC_RATIO)
constraint_count = INITIAL_POPULATION_SIZE - heuristic_count

population = (
    [create_heuristic_individual() for _ in range(heuristic_count)] +
    [create_constraint_satisfying_individual() for _ in range(constraint_count)]
)

# Evaluate using Hybrid Fitness
fitness_scores = [hybrid_fitness(ind) for ind in population]


# # === Parent Selection: Tournament ===
# def tournament_selection(pop, k=3):
#     selected = random.sample(pop, k)
#     selected.sort(key=hybrid_fitness, reverse=True)
#     return selected[0]

def tournament_selection(pop, k=3, num_parents=6):
    parents = []
    for _ in range(num_parents):
        selected = random.sample(pop, k)
        selected.sort(key=hybrid_fitness, reverse=True)
        parents.append(selected[0])
    return parents

# # === Parent Selection: Roulette Wheel ===
# def roulette_wheel_selection(pop):
#     fitnesses = [hybrid_fitness(ind) for ind in pop]
#     total_fitness = sum(fitnesses)
#     pick = random.uniform(0, total_fitness)
#     current = 0
#     for individual, fit in zip(pop, fitnesses):
#         current += fit
#         if current > pick:
#             return individual

def roulette_wheel_selection(pop, num_parents=6):
    fitnesses = [hybrid_fitness(ind) for ind in pop]
    total_fitness = sum(fitnesses)
    parents = []
    for _ in range(num_parents):
        pick = random.uniform(0, total_fitness)
        current = 0
        for individual, fit in zip(pop, fitnesses):
            current += fit
            if current > pick:
                parents.append(individual)
                break
    return parents


# Sigle poinnt crossover
def single_point_crossover(parent1, parent2):
    courses = list(parent1.keys())
    length = len(courses)
    k = random.randint(1, length - 2)

    offspring1 = {}
    offspring2 = {}

    for course in courses[:k]:
        offspring1[course] = parent1[course]
        offspring2[course] = parent2[course]

    for course in courses[k:]:
        offspring1[course] = parent2[course]
        offspring2[course] = parent1[course]

    return offspring1, offspring2


# Uniform Crossover
def Uniform_Crossover(parent1, parent2):
    p = 0.5
    offspring1 = {}
    offspring2 = {}

    for course in parent1:
        r = random.random()
        if r < p:
            offspring1[course] = parent1[course]
            offspring2[course] = parent2[course]
        else:
            offspring1[course] = parent2[course]
            offspring2[course] = parent1[course]
    return offspring1, offspring2


# Swap Mutation
def Swap_Mutation(individual):
    courses = list(individual.keys())
    length = len(courses)

    if length < 2:
        return individual

    r1 = random.randint(0, length - 1)
    r2 = random.randint(0, length - 1)

    while r1 == r2:  # Ensure two different courses
        r2 = random.randint(0, length - 1)

    course1 = courses[r1]
    course2 = courses[r2]

    individual[course1], individual[course2] = individual[course2], individual[course1]

    return individual


# Time Slot Mutation
def TimeslotMutation(individual):
    courses = list(individual.keys())
    length = len(courses)

    r = random.randint(0, length - 1)
    course = courses[r]

    _, room = individual[course]
    new_time = random.choice(timeslots)

    individual[course] = (new_time, room)

    return individual


def Order_Crossover(parent1, parent2):
    offspring1 = {}
    offspring2 = {}

    courses = list(parent1.keys())
    length = len(courses)

    r1 = random.randint(0, length - 1)
    r2 = random.randint(0, length - 1)

    while r1 == r2:
        r2 = random.randint(0, length - 1)
    if r1 > r2:
        r1, r2 = r2, r1

    courses1 = list(parent1.keys())
    courses2 = list(parent2.keys())

    for i in range(r1, r2 + 1):
        course = courses1[i]
        offspring1[course] = parent1[course]
        offspring2[course] = parent2[course]

    for i in list(range(r2 + 1, length)) + list(range(0, r2 + 1)):
        course_from_p2 = courses2[i]
        if course_from_p2 not in offspring1:
            offspring1[course_from_p2] = parent2[course_from_p2]

        course_from_p1 = courses1[i]
        if course_from_p1 not in offspring2:
            offspring2[course_from_p1] = parent1[course_from_p1]

    return offspring1, offspring2


# Scramble Mutation

def Scramble_Mutation(individual):
    courses = list(individual.keys())
    length = len(courses)

    r1 = random.randint(0, length - 1)
    r2 = random.randint(0, length - 1)

    while r1 == r2:  # Ensure two different courses
        r2 = random.randint(0, length - 1)

    if r1 > r2:
        r1, r2 = r2, r1

    selected_courses = courses[r1:r2 + 1]

    values_to_shuffle = [individual[course] for course in selected_courses]
    random.shuffle(values_to_shuffle)

    for course, new_value in zip(selected_courses, values_to_shuffle):
        individual[course] = new_value

    return individual


# Stagnation
def Stagnation(current_fitness, previous_fitness):
    global no_improvement_counter

    if current_fitness > previous_fitness:

        no_improvement_counter = 0

    else:

        no_improvement_counter += 1

    return no_improvement_counter


# survivor selection :elitism

def elitism_survivor_selection(population, offspring, elite_count):
    elite_count = min(elite_count, len(population))

    population_fitness = [(ind, hybrid_fitness(ind)) for ind in population]
    offspring_fitness = [(ind, hybrid_fitness(ind)) for ind in offspring]

    population_sorted = sorted(population_fitness, key=lambda x: x[1], reverse=True)
    offspring_sorted = sorted(offspring_fitness, key=lambda x: x[1], reverse=True)

    elites = [ind for ind, fit in population_sorted[:elite_count]]

    remaining_needed = len(population) - elite_count
    top_offspring = [ind for ind, fit in offspring_sorted[:remaining_needed]]

    new_population = elites + top_offspring
    return new_population


# steady_state_survivor_selection

def steady_state_survivor_selection(population, offspring, num_replacements):
    num_replacements = min(num_replacements, len(population))

    offspring_fitness = [(off, hybrid_fitness(off)) for off in offspring]

    population_fitness = [(ind, hybrid_fitness(ind)) for ind in population]

    population_fitness.sort(key=lambda x: x[1])

    worst_individuals = [ind for ind, fit in population_fitness[:num_replacements]]

    #  Compare each offspring with the worst individuals in population
    for off, off_fit in offspring_fitness:
        for worst_individual in worst_individuals:
            if off_fit > hybrid_fitness(worst_individual):
                population.remove(worst_individual)
                population.append(off)

    return population


# preserving diversity :fitness sharing

# calculate distance between two individuals
def timetable_distance(t1, t2):
    diff_count = 0
    for course in t1:
        if t1[course] != t2[course]:
            diff_count += 1
    return diff_count / len(t1)


# sharing function
def sharing_function(distance, sigma_share):
    return 1 - (distance / sigma_share) if distance <= sigma_share else 0


# apply fitness sharing
def apply_fitness_sharing(population, sigma_share=0.3):
    adjusted_fitnesses = []
    raw_fitnesses = [hybrid_fitness(ind) for ind in population]

    for i, ind_i in enumerate(population):
        niche_count = 0
        for j, ind_j in enumerate(population):
            dist = timetable_distance(ind_i, ind_j)
            niche_count += sharing_function(dist, sigma_share)

        # to avoid dividing on zero
        if niche_count == 0:
            niche_count = 1

        shared_fit = raw_fitnesses[i] / niche_count
        adjusted_fitnesses.append(shared_fit)

    return adjusted_fitnesses


# Generate a new individual timetable
# timetable = create_constraint_satisfying_individual()
# Print it
# print_timetable(timetable)


# === Step 9: Parameter Control and Tuning (Fully Enhanced) ===

# Initial parameters
mutation_rate = 0.2
crossover_rate = 0.8
population_size = 100
number_of_generations = 50


# Adaptive mutation rate
def adaptive_mutation_rate(current_generation, max_generations, initial_rate=0.2, final_rate=0.05):
    decay = (initial_rate - final_rate) / max_generations
    return max(final_rate, initial_rate - decay * current_generation)


# Adaptive crossover rate
def adaptive_crossover_rate(current_generation, max_generations, initial_rate=0.8, final_rate=0.9):
    growth = (final_rate - initial_rate) / max_generations
    return min(final_rate, initial_rate + growth * current_generation)


# Adaptive population size (optional: gradually decrease population to speed up late evolution)
def adaptive_population_size(current_generation, max_generations, initial_size=150, final_size=50):
    decay = (initial_size - final_size) / max_generations
    return max(final_size, int(initial_size - decay * current_generation))


# Adaptive number of generations (optional: useful for dynamic stopping based on progress)
def adaptive_number_of_generations(initial_generations=100, min_generations=50, improvement_rate=0.01,
                                   stagnation_counter=0):
    """Decrease the number of allowed generations if stagnation happens."""
    if stagnation_counter >= 10:
        return max(min_generations, initial_generations - 10)
    else:
        return initial_generations


# === Genetic Algorithm for Parameter Tuning (Fully Enhanced) ===

def tune_parameters_with_ga(parameter_population_size=10, generations=20):
    def generate_param_set():
        return {
            'mutation_rate': random.uniform(0.05, 0.3),
            'crossover_rate': random.uniform(0.6, 0.95),
            'population_size': random.choice([50, 100, 150]),
            'number_of_generations': random.choice([50, 100, 150])
        }

    def evaluate_param_set(params):
        # Create a small temp population based on the parameter set
        temp_population = [create_heuristic_individual() for _ in range(params['population_size'] // 20)]
        fitnesses = [hybrid_fitness(ind) for ind in temp_population]
        return sum(fitnesses) / len(fitnesses)

    population = [generate_param_set() for _ in range(parameter_population_size)]

    for gen in range(generations):
        scored = [(p, evaluate_param_set(p)) for p in population]
        scored.sort(key=lambda x: x[1], reverse=True)
        best = scored[:parameter_population_size // 2]

        new_population = [p for p, _ in best]
        while len(new_population) < parameter_population_size:
            p1, p2 = random.sample(new_population, 2)
            child = {
                'mutation_rate': (p1['mutation_rate'] + p2['mutation_rate']) / 2 + random.uniform(-0.01, 0.01),
                'crossover_rate': (p1['crossover_rate'] + p2['crossover_rate']) / 2 + random.uniform(-0.01, 0.01),
                'population_size': random.choice([50, 100, 150]),
                'number_of_generations': random.choice([50, 100, 150])
            }
            child['mutation_rate'] = min(max(child['mutation_rate'], 0.05), 0.3)
            child['crossover_rate'] = min(max(child['crossover_rate'], 0.6), 0.95)
            new_population.append(child)

        population = new_population

    best_params = max(population, key=evaluate_param_set)
    print("[Tuning] Best Parameters Found:", best_params)
    return best_params


# # === Step 10: Cooperative Co-evolution ===
#
# # Population A: evolves times
# def create_times_individual():
#     return {course: random.choice(timeslots) for course in courses}
#
#
# # Population B: evolves rooms
# def create_rooms_individual():
#     return {course: random.choice(rooms) for course in courses}
#
#
# # Combine a times individual and a rooms individual into a complete timetable
# def combine_times_and_rooms(times_individual, rooms_individual):
#     timetable = {}
#     for course in courses:
#         timetable[course] = (times_individual[course], rooms_individual[course])
#     return timetable
#
#
# # Fitness evaluation for combined individuals
# def coevolution_fitness(times_individual, rooms_individual):
#     timetable = combine_times_and_rooms(times_individual, rooms_individual)
#     return hybrid_fitness(timetable)
#
#
# # Initialize cooperative populations
# def initialize_cooperative_populations(pop_size):
#     times_population = [create_times_individual() for _ in range(pop_size)]
#     rooms_population = [create_rooms_individual() for _ in range(pop_size)]
#     return times_population, rooms_population
#
#
# # Select parents separately for times and rooms
# def tournament_selection_individual(population, k=3):
#     selected = random.sample(population, k)
#     selected.sort(key=lambda ind: hybrid_fitness(combine_times_and_rooms(ind, random.choice(population))), reverse=True)
#     return selected[0]
#
#
# # Evolve cooperative populations
# def evolve_cooperative_populations(times_population, rooms_population, generations=50):
#     pop_size = len(times_population)
#     for gen in range(generations):
#         new_times = []
#         new_rooms = []
#         for _ in range(pop_size // 2):
#             # Select parents
#             parent_times1 = tournament_selection_individual(times_population)
#             parent_times2 = tournament_selection_individual(times_population)
#             parent_rooms1 = tournament_selection_individual(rooms_population)
#             parent_rooms2 = tournament_selection_individual(rooms_population)
#
#             # Crossover (swap some genes)
#             offspring_times1, offspring_times2 = Uniform_Crossover(parent_times1, parent_times2)
#             offspring_rooms1, offspring_rooms2 = Uniform_Crossover(parent_rooms1, parent_rooms2)
#
#             # Mutation
#             offspring_times1 = TimeslotMutation(offspring_times1)
#             offspring_times2 = TimeslotMutation(offspring_times2)
#             offspring_rooms1 = Swap_Mutation(offspring_rooms1)
#             offspring_rooms2 = Swap_Mutation(offspring_rooms2)
#
#             new_times.extend([offspring_times1, offspring_times2])
#             new_rooms.extend([offspring_rooms1, offspring_rooms2])
#
#         # Update populations
#         times_population = new_times
#         rooms_population = new_rooms
#
#         # (Optional) Print best fitness at each generation
#         best_combined = max(
#             [(t, r) for t in times_population for r in rooms_population],
#             key=lambda pair: coevolution_fitness(pair[0], pair[1])
#         )
#         best_fitness = coevolution_fitness(best_combined[0], best_combined[1])
#         print(f"Generation {gen}: Best Cooperative Fitness = {best_fitness}")
#
#     return times_population, rooms_population


# === End of Step 10 ===

# Debug Output
print("Best fitness score in initial population:", min(fitness_scores))
print("Code executed successfully ✅")



FileNotFoundError: [Errno 2] No such file or directory: 'schedule.csv'